# Autovalidación de `salida.csv` del proyecto Ames

Este notebook permite que cada equipo revise su archivo antes de entregarlo. No modifica
`salida.csv`, no corrige encabezados y no calcula una nota. Las pruebas están organizadas
según los siete criterios de la rúbrica: dimensiones, faltantes, nombres, transformaciones
obligatorias, transformaciones opcionales, variables de calidad y datos de MongoDB.

Hay dos niveles de pruebas:

1. **Solo salida:** se ejecutan con `salida.csv`.
2. **Con fuentes:** comparan por `PID` con los archivos locales de `sources/`.

Los conteos que se muestran son resultados de la ejecución actual; el notebook no contiene
resultados prefabricados ni credenciales.

## 1. Configuración

Ajuste únicamente esta celda. Elija una convención y escriba exactamente uno de estos
valores: `camelCase`, `snake_case` o `kebab-case`.

Las siglas `PID`, `MS` y `QC` permanecen en mayúsculas en las tres convenciones. El notebook
compara los encabezados originales y su orden; nunca los renombra ni elimina espacios para
hacer que una prueba pase.

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import json
import os
import re

import pandas as pd
from IPython.display import display, Markdown


def encontrar_raiz_proyecto(inicio=None):
    # Busca la carpeta que contiene sources/AmesProperty.csv.
    inicio = Path(inicio or Path.cwd()).resolve()
    for candidato in (inicio, *inicio.parents):
        if (candidato / "sources" / "AmesProperty.csv").exists():
            return candidato
    # Permite ejecutar desde la carpeta notebooks aunque el servidor se haya iniciado allí.
    if (inicio.parent / "sources" / "AmesProperty.csv").exists():
        return inicio.parent
    return inicio


REPO_ROOT = Path(os.getenv("AMES_REPO_ROOT", encontrar_raiz_proyecto())).resolve()
SALIDA_PATH = Path(os.getenv("AMES_SALIDA_PATH", REPO_ROOT / "out" / "salida.csv")).resolve()
SOURCES_DIR = Path(os.getenv("AMES_SOURCES_DIR", REPO_ROOT / "sources")).resolve()

# Valores permitidos: camelCase, snake_case, kebab-case
NAMING_STYLE = os.getenv("AMES_NAMING_STYLE", "snake_case")

# None activa detección automática entre ; , tabulador y |.
DELIMITER = os.getenv("AMES_DELIMITER") or None
ENCODING = os.getenv("AMES_ENCODING", "utf-8-sig")
RUN_SOURCE_CHECKS = os.getenv("AMES_RUN_SOURCE_CHECKS", "true").lower() in {"1", "true", "yes", "si", "sí"}
MAX_EXAMPLES = 10

print("Raíz del proyecto:", REPO_ROOT)
print("Archivo de salida:", SALIDA_PATH)
print("Fuentes:", SOURCES_DIR)
print("Convención seleccionada:", NAMING_STYLE)
print("Comparación con fuentes:", "activada" if RUN_SOURCE_CHECKS else "desactivada")

## 2. Contrato de los 81 encabezados

La lista siguiente representa los campos y el orden definidos para el proyecto. La
convención elegida se aplica a esa lista antes de comparar. Solo `PID`, `MS` y `QC` son
excepciones en mayúsculas; los demás componentes siguen la convención seleccionada.

In [ ]:
FIELD_SPECS = [
    ("pid", ("PID",), "id"),
    ("ms_sub_class", ("MS", "sub", "class"), "num"),
    ("ms_zoning", ("MS", "zoning"), "text"),
    ("lot_frontage", ("lot", "frontage"), "num"),
    ("lot_area", ("lot", "area"), "num"),
    ("street", ("street",), "text"),
    ("alley", ("alley",), "text"),
    ("lot_shape", ("lot", "shape"), "text"),
    ("land_contour", ("land", "contour"), "text"),
    ("utilities", ("utilities",), "text"),
    ("lot_config", ("lot", "config"), "text"),
    ("land_slope", ("land", "slope"), "text"),
    ("neighborhood", ("neighborhood",), "text"),
    ("condition_1", ("condition", "1"), "text"),
    ("condition_2", ("condition", "2"), "text"),
    ("bldg_type", ("bldg", "type"), "text"),
    ("house_style", ("house", "style"), "text"),
    ("overall_qual", ("overall", "qual"), "num"),
    ("overall_cond", ("overall", "cond"), "num"),
    ("year_built", ("year", "built"), "num"),
    ("year_remod_add", ("year", "remod", "add"), "num"),
    ("roof_style", ("roof", "style"), "text"),
    ("roof_matl", ("roof", "matl"), "text"),
    ("exterior_1st", ("exterior", "1st"), "text"),
    ("exterior_2nd", ("exterior", "2nd"), "text"),
    ("mas_vnr_type", ("mas", "vnr", "type"), "text"),
    ("mas_vnr_area", ("mas", "vnr", "area"), "num"),
    ("exter_qual", ("exter", "qual"), "text"),
    ("exter_cond", ("exter", "cond"), "text"),
    ("foundation", ("foundation",), "text"),
    ("bsmt_qual", ("bsmt", "qual"), "text"),
    ("bsmt_cond", ("bsmt", "cond"), "text"),
    ("bsmt_exposure", ("bsmt", "exposure"), "text"),
    ("bsmt_fin_type_1", ("bsmt", "fin", "type", "1"), "text"),
    ("bsmt_fin_sf_1", ("bsmt", "fin", "sf", "1"), "num"),
    ("bsmt_fin_type_2", ("bsmt", "fin", "type", "2"), "text"),
    ("bsmt_fin_sf_2", ("bsmt", "fin", "sf", "2"), "num"),
    ("bsmt_unf_sf", ("bsmt", "unf", "sf"), "num"),
    ("total_bsmt_sf", ("total", "bsmt", "sf"), "num"),
    ("heating", ("heating",), "text"),
    ("heating_qc", ("heating", "QC"), "text"),
    ("central_air", ("central", "air"), "text"),
    ("electrical", ("electrical",), "text"),
    ("first_flr_sf", ("1st", "flr", "sf"), "num"),
    ("second_flr_sf", ("2nd", "flr", "sf"), "num"),
    ("low_qual_fin_sf", ("low", "qual", "fin", "sf"), "num"),
    ("gr_liv_area", ("gr", "liv", "area"), "num"),
    ("bsmt_full_bath", ("bsmt", "full", "bath"), "num"),
    ("bsmt_half_bath", ("bsmt", "half", "bath"), "num"),
    ("full_bath", ("full", "bath"), "num"),
    ("half_bath", ("half", "bath"), "num"),
    ("bedroom", ("bedroom",), "num"),
    ("kitchen_abv_gr", ("kitchen", "abv", "gr"), "num"),
    ("kitchen_qual", ("kitchen", "qual"), "text"),
    ("tot_rms_abv_grd", ("tot", "rms", "abv", "grd"), "num"),
    ("functional", ("functional",), "text"),
    ("fireplaces", ("fireplaces",), "num"),
    ("fireplace_qu", ("fireplace", "qu"), "text"),
    ("garage_type", ("garage", "type"), "text"),
    ("garage_yr_blt", ("garage", "yr", "blt"), "num"),
    ("garage_finish", ("garage", "finish"), "text"),
    ("garage_cars", ("garage", "cars"), "num"),
    ("garage_area", ("garage", "area"), "num"),
    ("garage_qual", ("garage", "qual"), "text"),
    ("garage_cond", ("garage", "cond"), "text"),
    ("paved_drive", ("paved", "drive"), "text"),
    ("wood_deck_sf", ("wood", "deck", "sf"), "num"),
    ("open_porch_sf", ("open", "porch", "sf"), "num"),
    ("enclosed_porch", ("enclosed", "porch"), "num"),
    ("three_ssn_porch", ("3ssn", "porch"), "num"),
    ("screen_porch", ("screen", "porch"), "num"),
    ("pool_area", ("pool", "area"), "num"),
    ("pool_qc", ("pool", "QC"), "text"),
    ("fence", ("fence",), "text"),
    ("misc_feature", ("misc", "feature"), "text"),
    ("misc_val", ("misc", "val"), "num"),
    ("mo_sold", ("mo", "sold"), "num"),
    ("yr_sold", ("yr", "sold"), "num"),
    ("sale_type", ("sale", "type"), "text"),
    ("sale_condition", ("sale", "condition"), "text"),
    ("sale_price", ("sale", "price"), "num"),
]

PROTECTED_ACRONYMS = {"PID", "MS", "QC"}
VALID_STYLES = {"camelCase", "snake_case", "kebab-case"}


def formatear_encabezado(tokens, style):
    if style not in VALID_STYLES:
        raise ValueError(f"Convención no válida: {style}. Use camelCase, snake_case o kebab-case.")

    def base(token):
        return token if token in PROTECTED_ACRONYMS else token.lower()

    if style == "snake_case":
        return "_".join(base(token) for token in tokens)
    if style == "kebab-case":
        return "-".join(base(token) for token in tokens)

    first = base(tokens[0])
    rest = []
    for token in tokens[1:]:
        if token in PROTECTED_ACRONYMS:
            rest.append(token)
        else:
            value = token.lower()
            rest.append(value[:1].upper() + value[1:])
    return first + "".join(rest)


if NAMING_STYLE not in VALID_STYLES:
    raise ValueError("NAMING_STYLE debe ser camelCase, snake_case o kebab-case.")

EXPECTED_BY_KEY = {
    key: formatear_encabezado(tokens, NAMING_STYLE)
    for key, tokens, _kind in FIELD_SPECS
}
KIND_BY_KEY = {key: kind for key, _tokens, kind in FIELD_SPECS}
EXPECTED_HEADERS = [EXPECTED_BY_KEY[key] for key, _tokens, _kind in FIELD_SPECS]

assert len(EXPECTED_HEADERS) == 81
assert len(set(EXPECTED_HEADERS)) == 81

display(pd.DataFrame({
    "posición": range(1, 82),
    "campo_interno": [key for key, _tokens, _kind in FIELD_SPECS],
    f"encabezado_{NAMING_STYLE}": EXPECTED_HEADERS,
    "tipo": [kind for _key, _tokens, kind in FIELD_SPECS],
}))

## 3. Carga y marco de resultados

`NA` se conserva como texto para poder distinguirlo de un nulo real. Los estados posibles
son `APROBADO`, `ERROR`, `ADVERTENCIA` y `NO EVALUADO`.

In [ ]:
RESULTS = []

CRITERIA = [
    "1. Dimensiones correctas",
    "2. Ausencia de nulos",
    "3. Nombres de columnas",
    "4. Transformaciones obligatorias",
    "5. Transformaciones opcionales",
    "6. Variables de calidad",
    "7. Extracción de MongoDB",
]


def registrar(criterio, prueba, estado, resumen, ejemplos=None):
    if estado not in {"APROBADO", "ERROR", "ADVERTENCIA", "NO EVALUADO"}:
        raise ValueError(f"Estado desconocido: {estado}")
    if ejemplos is None:
        ejemplos = []
    elif not isinstance(ejemplos, (list, tuple, set)):
        ejemplos = [ejemplos]
    ejemplos = [str(x) for x in list(ejemplos)[:MAX_EXAMPLES]]
    RESULTS.append({
        "criterio": criterio,
        "prueba": prueba,
        "estado": estado,
        "resumen": str(resumen),
        "ejemplos_PID": ", ".join(ejemplos),
    })


def detectar_delimitador(path):
    if DELIMITER:
        return DELIMITER
    with open(path, "r", encoding=ENCODING, newline="") as stream:
        sample = stream.read(8192)
    try:
        return csv.Sniffer().sniff(sample, delimiters=";,\t|").delimiter
    except csv.Error:
        return ";"


df = pd.DataFrame()
OUTPUT_READY = False
USED_DELIMITER = None

if not SALIDA_PATH.exists():
    registrar(
        CRITERIA[0], "Carga de salida.csv", "ERROR",
        f"No se encontró el archivo: {SALIDA_PATH}",
    )
else:
    try:
        USED_DELIMITER = detectar_delimitador(SALIDA_PATH)
        df = pd.read_csv(
            SALIDA_PATH,
            sep=USED_DELIMITER,
            dtype=str,
            keep_default_na=False,
            encoding=ENCODING,
        )
        OUTPUT_READY = True
        registrar(
            CRITERIA[0], "Carga de salida.csv", "APROBADO",
            f"Archivo leído con separador {USED_DELIMITER!r}: {df.shape[0]} filas y {df.shape[1]} columnas.",
        )
        display(df.head(3))
    except Exception as exc:
        registrar(CRITERIA[0], "Carga de salida.csv", "ERROR", f"No fue posible leer el archivo: {exc}")


def encabezado(key):
    return EXPECTED_BY_KEY[key]


def faltan_columnas(keys):
    if not OUTPUT_READY:
        return [encabezado(key) for key in keys]
    return [encabezado(key) for key in keys if encabezado(key) not in df.columns]


def serie_numerica(key):
    return pd.to_numeric(df[encabezado(key)].astype(str).str.strip(), errors="coerce")


def normalizar_pid(value):
    return str(value).strip()


def comparar_numeros(actual, esperado, tolerancia=1e-9):
    actual = pd.to_numeric(pd.Series(actual), errors="coerce")
    esperado = pd.to_numeric(pd.Series(esperado), errors="coerce")
    return actual.notna() & esperado.notna() & ((actual - esperado).abs() <= tolerancia)

## 4. Criterios 1 a 3: dimensiones, faltantes y encabezados

In [ ]:
if OUTPUT_READY:
    # 1. Dimensiones y PID
    expected_shape = (2930, 81)
    actual_shape = df.shape
    registrar(
        CRITERIA[0], "Dimensiones 2930 × 81",
        "APROBADO" if actual_shape == expected_shape else "ERROR",
        f"Esperado: {expected_shape}; recibido: {actual_shape}.",
    )

    pid_col = encabezado("pid")
    if pid_col not in df.columns:
        registrar(CRITERIA[0], "PID válido y único", "NO EVALUADO", f"Falta la columna {pid_col!r}.")
    else:
        pid_raw = df[pid_col].astype(str).str.strip()
        invalid_pid = ~pid_raw.str.fullmatch(r"[1-9]\d*")
        duplicate_pid = pid_raw.duplicated(keep=False)
        bad = df.loc[invalid_pid | duplicate_pid, pid_col].tolist()
        registrar(
            CRITERIA[0], "PID válido y único",
            "APROBADO" if not bad else "ERROR",
            f"PID inválidos: {int(invalid_pid.sum())}; filas con PID duplicado: {int(duplicate_pid.sum())}.",
            bad,
        )

    # 3. Encabezados exactos y orden
    actual_headers = list(df.columns)
    header_mismatches = []
    for position in range(max(len(EXPECTED_HEADERS), len(actual_headers))):
        expected = EXPECTED_HEADERS[position] if position < len(EXPECTED_HEADERS) else "<sin campo esperado>"
        actual = actual_headers[position] if position < len(actual_headers) else "<ausente>"
        if expected != actual:
            header_mismatches.append(f"{position + 1}: esperado={expected!r}, recibido={actual!r}")

    registrar(
        CRITERIA[2], f"Encabezados exactos en {NAMING_STYLE} y orden correcto",
        "APROBADO" if not header_mismatches else "ERROR",
        "Los 81 encabezados coinciden exactamente." if not header_mismatches
        else f"Se encontraron {len(header_mismatches)} diferencias de nombre u orden.",
        header_mismatches,
    )

    duplicate_headers = [name for name, count in Counter(actual_headers).items() if count > 1]
    registrar(
        CRITERIA[2], "Encabezados sin duplicados",
        "APROBADO" if not duplicate_headers else "ERROR",
        "No hay encabezados duplicados." if not duplicate_headers else f"Duplicados: {duplicate_headers}",
    )

    headers_with_spaces = [name for name in actual_headers if any(ch.isspace() for ch in str(name))]
    registrar(
        CRITERIA[2], "Encabezados sin espacios",
        "APROBADO" if not headers_with_spaces else "ERROR",
        "No se encontraron espacios." if not headers_with_spaces else f"Encabezados con espacios: {len(headers_with_spaces)}.",
        headers_with_spaces,
    )

    # 2. Marcadores y tipos
    # "None" es un valor categórico válido en Ames (por ejemplo, Mas Vnr Type),
    # por eso no se trata como marcador nulo. Los vacíos reales se leen como "".
    invalid_null_literals = {"", "null", "nan", "n/a", "<na>", "na"}
    invalid_marker_rows = []
    invalid_type_rows = []
    surrounding_whitespace_rows = []

    for key, _tokens, kind in FIELD_SPECS:
        column = encabezado(key)
        if column not in df.columns:
            continue
        raw = df[column].astype(str)
        stripped = raw.str.strip()

        whitespace_mask = raw.ne(stripped)
        surrounding_whitespace_rows.extend(
            f"{column} fila {index + 2}" for index in df.index[whitespace_mask][:MAX_EXAMPLES]
        )

        if kind == "text":
            invalid_literal = stripped.map(
                lambda value: value != "NA" and value.casefold() in invalid_null_literals
            )
            numeric_zero = stripped.eq("0")
            bad_mask = invalid_literal | numeric_zero
        else:
            invalid_literal = stripped.map(
                lambda value: value.casefold() in invalid_null_literals
            )
            parsed = pd.to_numeric(stripped, errors="coerce")
            bad_mask = invalid_literal | parsed.isna()

        invalid_marker_rows.extend(
            f"{column} fila {index + 2}" for index in df.index[invalid_literal][:MAX_EXAMPLES]
        )
        invalid_type_rows.extend(
            f"{column} fila {index + 2}" for index in df.index[bad_mask][:MAX_EXAMPLES]
        )

    registrar(
        CRITERIA[1], "Marcadores de faltantes",
        "APROBADO" if not invalid_marker_rows else "ERROR",
        "Solo se usan NA en cualitativos y valores numéricos —incluido 0— en numéricos."
        if not invalid_marker_rows else f"Se detectaron marcadores vacíos o no permitidos en al menos {len(invalid_marker_rows)} posiciones mostradas.",
        invalid_marker_rows,
    )
    registrar(
        CRITERIA[1], "Compatibilidad de tipos y marcadores",
        "APROBADO" if not invalid_type_rows else "ERROR",
        "Los campos cualitativos y numéricos usan representaciones compatibles."
        if not invalid_type_rows else f"Se detectaron valores incompatibles en al menos {len(invalid_type_rows)} posiciones mostradas.",
        invalid_type_rows,
    )
    registrar(
        CRITERIA[1], "Espacios alrededor de valores",
        "APROBADO" if not surrounding_whitespace_rows else "ADVERTENCIA",
        "No se detectaron espacios al inicio o final de valores."
        if not surrounding_whitespace_rows else "Hay valores con espacios al inicio o al final; pueden afectar comparaciones y dominios.",
        surrounding_whitespace_rows,
    )
else:
    for criterion, test in [
        (CRITERIA[0], "Dimensiones y PID"),
        (CRITERIA[1], "Marcadores de faltantes"),
        (CRITERIA[2], "Encabezados exactos"),
    ]:
        registrar(criterion, test, "NO EVALUADO", "No se cargó salida.csv.")

## 5. Transformaciones obligatorias verificables con la salida

Estas pruebas no reemplazan la comparación con los orígenes. Comprueban invariantes que
deben cumplirse dentro del archivo final.

In [ ]:
if OUTPUT_READY:
    required = ["first_flr_sf", "second_flr_sf", "low_qual_fin_sf", "gr_liv_area"]
    missing = faltan_columnas(required)
    if missing:
        registrar(CRITERIA[3], "Gr Liv Area por fila", "NO EVALUADO", f"Faltan columnas: {missing}")
    else:
        expected_gr = (
            serie_numerica("first_flr_sf")
            + serie_numerica("second_flr_sf")
            + serie_numerica("low_qual_fin_sf")
        )
        actual_gr = serie_numerica("gr_liv_area")
        good = comparar_numeros(actual_gr, expected_gr)
        bad_index = df.index[~good]
        pids = df.loc[bad_index, encabezado("pid")].tolist() if encabezado("pid") in df.columns else bad_index.tolist()
        registrar(
            CRITERIA[3], "Gr Liv Area por fila",
            "APROBADO" if len(bad_index) == 0 else "ERROR",
            f"Filas que no cumplen 1st + 2nd + Low Qual = Gr Liv Area: {len(bad_index)}.",
            pids,
        )

    required = ["year_built", "year_remod_add"]
    missing = faltan_columnas(required)
    if missing:
        registrar(CRITERIA[3], "Year Remod Add sin marcador 0", "NO EVALUADO", f"Faltan columnas: {missing}")
    else:
        remod = serie_numerica("year_remod_add")
        bad_mask = remod.isna() | remod.eq(0)
        pids = df.loc[bad_mask, encabezado("pid")].tolist() if encabezado("pid") in df.columns else df.index[bad_mask].tolist()
        registrar(
            CRITERIA[3], "Year Remod Add sin marcador 0",
            "APROBADO" if not bad_mask.any() else "ERROR",
            f"Filas donde Year Remod Add continúa vacío, no numérico o en 0: {int(bad_mask.sum())}.",
            pids,
        )
else:
    registrar(CRITERIA[3], "Transformaciones internas", "NO EVALUADO", "No se cargó salida.csv.")

## 6. Preparación de fuentes locales

Las celdas siguientes solo leen archivos dentro de `sources/`. Si una fuente no está
disponible, la prueba correspondiente queda como `NO EVALUADO`; el notebook no solicita
conexiones ni credenciales.

In [ ]:
BASE_SOURCE = SOURCES_DIR / "AmesProperty.csv"
SCRIPT_DIR = SOURCES_DIR / "scriptDatos"
MONGO_DIR = SOURCES_DIR / "mongoCollections"

base_df = pd.DataFrame()
BASE_READY = False

if RUN_SOURCE_CHECKS and BASE_SOURCE.exists():
    try:
        base_df = pd.read_csv(BASE_SOURCE, sep=";", dtype=str, keep_default_na=False, encoding="utf-8-sig")
        base_df["PID"] = base_df["PID"].astype(str).str.strip()
        BASE_READY = True
    except Exception as exc:
        registrar(CRITERIA[0], "Lectura de Ames Property.csv", "NO EVALUADO", f"Error al leer la fuente: {exc}")
elif RUN_SOURCE_CHECKS:
    registrar(CRITERIA[0], "Lectura de Ames Property.csv", "NO EVALUADO", f"No se encontró {BASE_SOURCE}")


def parse_sql_inserts(path, table_name):
    # Lee INSERT ... VALUES de los dumps suministrados; no ejecuta SQL.
    prefix = f"INSERT INTO public.{table_name} VALUES ("
    with open(path, "r", encoding="utf-8", errors="replace") as stream:
        for line in stream:
            line = line.strip()
            if not line.startswith(prefix) or not line.endswith(");"):
                continue
            payload = line[len(prefix):-2]
            yield next(csv.reader([payload], delimiter=",", quotechar="'", skipinitialspace=True))


def output_indexed():
    if not OUTPUT_READY or encabezado("pid") not in df.columns:
        return None
    result = df.copy()
    result[encabezado("pid")] = result[encabezado("pid")].map(normalizar_pid)
    if result[encabezado("pid")].duplicated().any():
        return None
    return result.set_index(encabezado("pid"), drop=False)


output_by_pid = output_indexed()

if OUTPUT_READY and BASE_READY and output_by_pid is not None:
    output_pids = set(output_by_pid.index)
    base_pids = set(base_df["PID"])
    missing_pids = sorted(base_pids - output_pids)
    extra_pids = sorted(output_pids - base_pids)
    registrar(
        CRITERIA[0], "Cobertura de PID frente al CSV base",
        "APROBADO" if not missing_pids and not extra_pids else "ERROR",
        f"PID faltantes: {len(missing_pids)}; PID adicionales: {len(extra_pids)}.",
        missing_pids + extra_pids,
    )
elif RUN_SOURCE_CHECKS:
    registrar(CRITERIA[0], "Cobertura de PID frente al CSV base", "NO EVALUADO", "No están disponibles salida, PID único o CSV base.")

## 7. Transformaciones obligatorias contra las fuentes

In [ ]:
# Year Remod/Add: comparar con el CSV original por PID.
if RUN_SOURCE_CHECKS and OUTPUT_READY and BASE_READY and output_by_pid is not None:
    key = "year_remod_add"
    column = encabezado(key)
    if column not in df.columns:
        registrar(CRITERIA[3], "Year Remod Add contra la fuente", "NO EVALUADO", f"Falta la columna {column!r}.")
    else:
        source = base_df.set_index("PID")
        expected = source["Year Remod/Add"].astype(str).str.strip()
        built = source["Year Built"].astype(str).str.strip()
        expected = expected.where(expected.ne(""), built)
        common = output_by_pid.index.intersection(expected.index)
        actual = pd.to_numeric(output_by_pid.loc[common, column].astype(str).str.strip(), errors="coerce")
        expected_num = pd.to_numeric(expected.loc[common], errors="coerce")
        good = comparar_numeros(actual.reset_index(drop=True), expected_num.reset_index(drop=True))
        bad_pids = list(common[~good.to_numpy()])
        registrar(
            CRITERIA[3], "Year Remod Add contra la fuente",
            "APROBADO" if not bad_pids else "ERROR",
            f"PID con valor distinto al original o a Year Built cuando faltaba remodelación: {len(bad_pids)}.",
            bad_pids,
        )
elif RUN_SOURCE_CHECKS:
    registrar(CRITERIA[3], "Year Remod Add contra la fuente", "NO EVALUADO", "No están disponibles salida, PID único o CSV base.")

# FloorDetail: sumar habitaciones y baños por PID.
floor_path = SCRIPT_DIR / "floordetail.sql"
floor_keys = ["bedroom", "full_bath", "half_bath"]
if RUN_SOURCE_CHECKS and OUTPUT_READY and output_by_pid is not None and floor_path.exists() and not faltan_columnas(floor_keys):
    rows = []
    try:
        for values in parse_sql_inserts(floor_path, "floordetail"):
            if len(values) >= 5:
                rows.append({
                    "PID": normalizar_pid(values[0]),
                    "bedroom": float(values[2]),
                    "full_bath": float(values[3]),
                    "half_bath": float(values[4]),
                })
        floor = pd.DataFrame(rows).groupby("PID", as_index=True)[floor_keys].sum()
        common = output_by_pid.index.intersection(floor.index)
        bad_by_field = {}
        bad_all = set()
        for key in floor_keys:
            actual = pd.to_numeric(output_by_pid.loc[common, encabezado(key)].astype(str).str.strip(), errors="coerce")
            expected = floor.loc[common, key]
            good = comparar_numeros(actual.reset_index(drop=True), expected.reset_index(drop=True))
            bad = list(common[~good.to_numpy()])
            bad_by_field[encabezado(key)] = len(bad)
            bad_all.update(bad)
        registrar(
            CRITERIA[3], "Agregación de FloorDetail por PID",
            "APROBADO" if not bad_all else "ERROR",
            f"Diferencias por campo: {bad_by_field}.",
            sorted(bad_all),
        )
    except Exception as exc:
        registrar(CRITERIA[3], "Agregación de FloorDetail por PID", "NO EVALUADO", f"No se pudo procesar la fuente: {exc}")
elif RUN_SOURCE_CHECKS:
    registrar(CRITERIA[3], "Agregación de FloorDetail por PID", "NO EVALUADO", "Falta la salida, PID único, columnas requeridas o floordetail.sql.")

# SaleProperty: derivar mes y año de la fecha de venta.
sale_path = SCRIPT_DIR / "saleproperty.sql"
sale_keys = ["mo_sold", "yr_sold"]
if RUN_SOURCE_CHECKS and OUTPUT_READY and output_by_pid is not None and sale_path.exists() and not faltan_columnas(sale_keys):
    expected_sale = {}
    try:
        for values in parse_sql_inserts(sale_path, "saleproperty"):
            if len(values) >= 2:
                pid = normalizar_pid(values[0])
                date_text = values[1].strip()
                match = re.match(r"(\d{4})-(\d{2})-", date_text)
                if match:
                    expected_sale[pid] = {"yr_sold": int(match.group(1)), "mo_sold": int(match.group(2))}
        sale = pd.DataFrame.from_dict(expected_sale, orient="index")
        common = output_by_pid.index.intersection(sale.index)
        bad_all = set()
        bad_by_field = {}
        for key in sale_keys:
            actual = pd.to_numeric(output_by_pid.loc[common, encabezado(key)].astype(str).str.strip(), errors="coerce")
            expected = sale.loc[common, key]
            good = comparar_numeros(actual.reset_index(drop=True), expected.reset_index(drop=True))
            bad = list(common[~good.to_numpy()])
            bad_by_field[encabezado(key)] = len(bad)
            bad_all.update(bad)
        registrar(
            CRITERIA[3], "Mes y año derivados de Sale Date",
            "APROBADO" if not bad_all else "ERROR",
            f"Diferencias por campo: {bad_by_field}.",
            sorted(bad_all),
        )
    except Exception as exc:
        registrar(CRITERIA[3], "Mes y año derivados de Sale Date", "NO EVALUADO", f"No se pudo procesar la fuente: {exc}")
elif RUN_SOURCE_CHECKS:
    registrar(CRITERIA[3], "Mes y año derivados de Sale Date", "NO EVALUADO", "Falta la salida, PID único, columnas requeridas o saleproperty.sql.")

## 8. Transformaciones opcionales

Para cada columna se acepta uno de dos resultados completos: conservar exactamente los
valores originales por PID o transformar toda la columna a códigos. Una mezcla parcial de
ambas alternativas se reporta como error.

In [ ]:
NEIGHBORHOOD_CODES = {
    "Bloomington Heights": "Blmngtn", "Bluestem": "Blueste", "Briardale": "BrDale",
    "Brookside": "BrkSide", "Clear Creek": "ClearCr", "College Creek": "CollgCr",
    "Crawford": "Crawfor", "Edwards": "Edwards", "Gilbert": "Gilbert",
    "Greens": "Greens", "GrnHill": "GrnHill", "Iowa DOT and Rail Road": "IDOTRR",
    "Landmrk": "Landmrk", "Meadow Village": "MeadowV", "Mitchell": "Mitchel",
    "North Ames": "Names", "Northpark Villa": "NPkVill", "Northridge": "NoRidge",
    "Northridge Heights": "NridgHt", "Northwest Ames": "NWAmes", "Old Town": "OldTown",
    "Sawyer": "Sawyer", "Sawyer West": "SawyerW", "Somerset": "Somerst",
    "South & West of Iowa State University": "SWISU", "Stone Brook": "StoneBr",
    "Timberland": "Timber", "Veenker": "Veenker",
}
CONDITION_CODES = {
    "Adjacent to arterial street": "Artery", "Artery": "Artery",
    "Adjacent to feeder street": "Feedr", "Feedr": "Feedr",
    "Normal": "Norm", "Norm": "Norm",
    "Within 200' of North-South Railroad": "RRNn", "RRNn": "RRNn",
    "Adjacent to North-South Railroad": "RRAn", "RRAn": "RRAn",
    "Near positive off-site feature--park, greenbelt, etc.": "PosN", "PosN": "PosN",
    "Adjacent to postive off-site feature": "PosA", "PosA": "PosA",
    "Within 200' of East-West Railroad": "RRNe", "RRNe": "RRNe",
    "Adjacent to East-West Railroad": "RRAe", "RRAe": "RRAe",
}
LOT_SHAPE_CODES = {
    "Regular": "Reg", "Reg": "Reg",
    "Slightly irregular": "IR1", "IR1": "IR1",
    "Moderately Irregular": "IR2", "IR2": "IR2",
    "Irregular": "IR3", "IR3": "IR3",
}

optional_specs = [
    ("neighborhood", "Neighborhood", NEIGHBORHOOD_CODES),
    ("condition_1", "Condition 1", CONDITION_CODES),
    ("condition_2", "Condition 2", CONDITION_CODES),
    ("lot_shape", "Lot Shape", LOT_SHAPE_CODES),
]

if RUN_SOURCE_CHECKS and OUTPUT_READY and BASE_READY and output_by_pid is not None:
    source = base_df.set_index("PID")
    for key, source_col, mapping in optional_specs:
        output_col = encabezado(key)
        if output_col not in df.columns:
            registrar(CRITERIA[4], f"Opcional: {source_col}", "NO EVALUADO", f"Falta la columna {output_col!r}.")
            continue
        common = output_by_pid.index.intersection(source.index)
        actual = output_by_pid.loc[common, output_col].astype(str).str.strip()
        raw_expected = source.loc[common, source_col].astype(str).str.strip()
        code_expected = raw_expected.map(mapping)
        unknown_source = code_expected.isna()
        raw_good = actual.eq(raw_expected)
        code_good = actual.eq(code_expected)
        if raw_good.all():
            registrar(CRITERIA[4], f"Opcional: {source_col}", "APROBADO", "La columna conserva de forma consistente los valores originales.")
        elif not unknown_source.any() and code_good.all():
            registrar(CRITERIA[4], f"Opcional: {source_col}", "APROBADO", "La columna usa de forma consistente los códigos definidos.")
        else:
            bad_mask = ~(raw_good | code_good) | unknown_source
            bad_pids = list(common[bad_mask.to_numpy()])
            registrar(
                CRITERIA[4], f"Opcional: {source_col}", "ERROR",
                "La columna no coincide completamente con los valores originales ni con una transformación completa a códigos.",
                bad_pids,
            )
elif RUN_SOURCE_CHECKS:
    registrar(CRITERIA[4], "Transformaciones opcionales", "NO EVALUADO", "No están disponibles salida, PID único o CSV base.")
else:
    registrar(CRITERIA[4], "Transformaciones opcionales", "NO EVALUADO", "Las comparaciones con fuentes están desactivadas.")

## 9. Variables de calidad

In [ ]:
typequality_path = SCRIPT_DIR / "typequality.sql"
amesdb_path = SCRIPT_DIR / "amesdbtemp.sql"
quality_keys = ["exter_cond", "exter_qual", "heating_qc", "kitchen_qual"]

if (
    RUN_SOURCE_CHECKS and OUTPUT_READY and output_by_pid is not None
    and typequality_path.exists() and amesdb_path.exists()
    and not faltan_columnas(quality_keys)
):
    try:
        quality_map = {}
        for values in parse_sql_inserts(typequality_path, "typequality"):
            if len(values) >= 2:
                quality_map[str(values[0]).strip()] = str(values[1]).strip()

        expected_rows = {}
        for values in parse_sql_inserts(amesdb_path, "amesdbtemp"):
            if len(values) >= 6:
                pid = normalizar_pid(values[5])
                expected_rows[pid] = {
                    "exter_cond": quality_map.get(str(values[0]).strip()),
                    "exter_qual": quality_map.get(str(values[1]).strip()),
                    "heating_qc": quality_map.get(str(values[2]).strip()),
                    "kitchen_qual": quality_map.get(str(values[3]).strip()),
                }
        expected = pd.DataFrame.from_dict(expected_rows, orient="index")
        common = output_by_pid.index.intersection(expected.index)
        bad_all = set()
        bad_by_field = {}
        for key in quality_keys:
            actual = output_by_pid.loc[common, encabezado(key)].astype(str).str.strip()
            wanted = expected.loc[common, key].astype(str).str.strip()
            good = actual.eq(wanted)
            bad = list(common[~good.to_numpy()])
            bad_by_field[encabezado(key)] = len(bad)
            bad_all.update(bad)
        registrar(
            CRITERIA[5], "Códigos de calidad derivados de TypeQuality",
            "APROBADO" if not bad_all else "ERROR",
            f"Diferencias por campo: {bad_by_field}.",
            sorted(bad_all),
        )
    except Exception as exc:
        registrar(CRITERIA[5], "Códigos de calidad derivados de TypeQuality", "NO EVALUADO", f"No se pudieron procesar las fuentes: {exc}")
elif RUN_SOURCE_CHECKS:
    registrar(CRITERIA[5], "Códigos de calidad derivados de TypeQuality", "NO EVALUADO", "Faltan salida, PID único, columnas o scripts relacionales.")
else:
    registrar(CRITERIA[5], "Códigos de calidad derivados de TypeQuality", "NO EVALUADO", "Las comparaciones con fuentes están desactivadas.")

## 10. MongoDB por PID

Cada colección se compara por `PID`. Cuando no existe documento —o falta un campo— se
espera `NA` para variables cualitativas y `0` para variables numéricas. Antes de comparar se
verifica que la colección no tenga PID duplicados que puedan multiplicar propiedades.

In [ ]:
def unwrap_mongo(value):
    if value is None:
        return None
    if isinstance(value, dict):
        for key in ("$numberLong", "$numberInt", "$numberDouble", "$numberDecimal"):
            if key in value:
                return str(value[key])
    return value


MONGO_CONFIG = {
    "garage": {
        "path": MONGO_DIR / "test.garage.json",
        "fields": [
            ("garage_type", "Garage Type", "text"),
            ("garage_yr_blt", "Garage Yr Blt", "num"),
            ("garage_finish", "Garage Finish", "text"),
            ("garage_cars", "Garage Cars", "num"),
            ("garage_area", "Garage Area", "num"),
            ("garage_qual", "Garage Qual", "text"),
            ("garage_cond", "Garage Cond", "text"),
        ],
    },
    "pool": {
        "path": MONGO_DIR / "test.pool.json",
        "fields": [
            ("pool_area", "Pool Area", "num"),
            ("pool_qc", "Pool QC", "text"),
        ],
    },
    "bsmt": {
        "path": MONGO_DIR / "test.bsmt.json",
        "fields": [
            ("bsmt_qual", "Bsmt Qual", "text"),
            ("bsmt_cond", "Bsmt Cond", "text"),
            ("bsmt_exposure", "Bsmt Exposure", "text"),
            ("bsmt_fin_type_1", "BsmtFin Type 1", "text"),
            ("bsmt_fin_sf_1", "BsmtFin SF 1", "num"),
            ("bsmt_fin_type_2", "BsmtFin Type 2", "text"),
            ("bsmt_fin_sf_2", "BsmtFin SF 2", "num"),
            ("bsmt_unf_sf", "Bsmt Unf SF", "num"),
            ("total_bsmt_sf", "Total Bsmt SF", "num"),
            ("bsmt_full_bath", "Bsmt Full Bath", "num"),
            ("bsmt_half_bath", "Bsmt Half Bath", "num"),
        ],
    },
    "misc": {
        "path": MONGO_DIR / "test.misc.json",
        "fields": [
            ("misc_feature", "Misc Feature", "text"),
            ("misc_val", "Misc Val", "num"),
        ],
    },
}


if RUN_SOURCE_CHECKS and OUTPUT_READY and output_by_pid is not None:
    for collection, config in MONGO_CONFIG.items():
        path = config["path"]
        needed_keys = [key for key, _source_name, _kind in config["fields"]]
        missing_columns = faltan_columnas(needed_keys)
        if not path.exists() or missing_columns:
            registrar(
                CRITERIA[6], f"Colección {collection}", "NO EVALUADO",
                f"Fuente existente: {path.exists()}; columnas faltantes: {missing_columns}.",
            )
            continue

        try:
            with open(path, "r", encoding="utf-8") as stream:
                documents = json.load(stream)
            records = {}
            duplicate_pids = set()
            missing_source_pid = 0
            for document in documents:
                pid_value = unwrap_mongo(document.get("PID"))
                if pid_value is None:
                    missing_source_pid += 1
                    continue
                pid = normalizar_pid(pid_value)
                if pid in records:
                    duplicate_pids.add(pid)
                records[pid] = document

            if duplicate_pids or missing_source_pid:
                registrar(
                    CRITERIA[6], f"Cardinalidad de {collection}", "ERROR",
                    f"PID duplicados: {len(duplicate_pids)}; documentos sin PID: {missing_source_pid}.",
                    sorted(duplicate_pids),
                )
            else:
                registrar(
                    CRITERIA[6], f"Cardinalidad de {collection}", "APROBADO",
                    f"{len(documents)} documentos; ningún PID duplicado o ausente.",
                )

            bad_by_field = {}
            bad_all = set()
            for key, source_name, kind in config["fields"]:
                output_col = encabezado(key)
                bad = []
                for pid, row in output_by_pid.iterrows():
                    document = records.get(pid)
                    source_value = None if document is None else unwrap_mongo(document.get(source_name))
                    expected_value = ("0" if kind == "num" else "NA") if source_value is None else str(source_value).strip()
                    actual_value = str(row[output_col]).strip()
                    if kind == "num":
                        actual_num = pd.to_numeric(pd.Series([actual_value]), errors="coerce").iloc[0]
                        expected_num = pd.to_numeric(pd.Series([expected_value]), errors="coerce").iloc[0]
                        equal = pd.notna(actual_num) and pd.notna(expected_num) and abs(actual_num - expected_num) <= 1e-9
                    else:
                        equal = actual_value == expected_value
                    if not equal:
                        bad.append(pid)
                bad_by_field[output_col] = len(bad)
                bad_all.update(bad)

            registrar(
                CRITERIA[6], f"Valores de {collection} por PID",
                "APROBADO" if not bad_all else "ERROR",
                f"Diferencias por campo: {bad_by_field}.",
                sorted(bad_all),
            )
        except Exception as exc:
            registrar(CRITERIA[6], f"Colección {collection}", "NO EVALUADO", f"No se pudo procesar la fuente: {exc}")
elif RUN_SOURCE_CHECKS:
    registrar(CRITERIA[6], "Comparación de MongoDB", "NO EVALUADO", "No están disponibles salida o PID único.")
else:
    registrar(CRITERIA[6], "Comparación de MongoDB", "NO EVALUADO", "Las comparaciones con fuentes están desactivadas.")

## 11. Resumen según la rúbrica

Un criterio queda en `ERROR` si alguna de sus pruebas falla. `NO EVALUADO` indica que faltó
una fuente, una columna o el archivo de salida. Este resumen es diagnóstico: no asigna una
calificación y no reemplaza la revisión docente.

In [ ]:
results_df = pd.DataFrame(RESULTS)
if results_df.empty:
    display(Markdown("**No se ejecutó ninguna prueba.**"))
else:
    severity = {"APROBADO": 0, "ADVERTENCIA": 1, "NO EVALUADO": 2, "ERROR": 3}

    def criterion_status(group):
        return max(group["estado"], key=lambda value: severity[value])

    summary_rows = []
    for criterion in CRITERIA:
        group = results_df[results_df["criterio"] == criterion]
        status = "NO EVALUADO" if group.empty else criterion_status(group)
        summary_rows.append({
            "criterio de la rúbrica": criterion,
            "estado": status,
            "pruebas": len(group),
            "errores": int((group["estado"] == "ERROR").sum()) if not group.empty else 0,
        })

    rubric_summary = pd.DataFrame(summary_rows)
    display(Markdown("### Resumen"))
    display(rubric_summary)
    display(Markdown("### Detalle de pruebas"))
    display(results_df)

    print("\nImportante: corrija primero los ERROR y vuelva a ejecutar el notebook completo.")
    print("NO EVALUADO no significa aprobado; revise rutas, fuentes y encabezados.")